In [0]:
import mlflow.xgboost

def tool_register_and_deploy_model(model_results, validation_results, model_name="emissor_a_behavior_score"):
    """Registra o modelo no Unity Catalog do Databricks."""
    if not validation_results["approved"]:
        raise ValueError("Tentativa de deploy de um modelo reprovado pelo Quality Gate.")
        
    model = model_results["model"]
    
    with mlflow.start_run() as run:
        mlflow.log_metrics({"final_auc": validation_results["auc"], "final_ks": validation_results["ks"]})
        
        # Salvando e registrando no Unity Catalog (ex: catalog.schema.model_name)
        # uc_model_name = f"emissor_a.gold.{model_name}"
        mlflow.xgboost.log_model(
            model, 
            artifact_path="model",
            registered_model_name=model_name
        )
        
        model_uri = f"runs:/{run.info.run_id}/model"
        print(f"[Agente] Modelo registrado com sucesso! URI: {model_uri}")
        
    return {"status": "deployed", "model_uri": model_uri}